# Ningxia

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load raw data

In [ ]:
import pandas as pd
import numpy as np
import os

FILE_PATH = str(RAW_DIR / 'Photovoltaic power data.xlsx')
if not os.path.exists(FILE_PATH):
    for root, dirs, files in os.walk(str(RAW_DIR)):
        if 'Photovoltaic power data.xlsx' in files:
            FILE_PATH = os.path.join(root, 'Photovoltaic power data.xlsx')
            break

xls = pd.ExcelFile(FILE_PATH)
df = pd.read_excel(FILE_PATH, sheet_name=xls.sheet_names[0])

RENAME_MAPPING = {
    '采样结束时刻': 'timestamp_end',
    '采样起始时刻': 'timestamp_start',
    '时间间隔': 'time_interval',
    '气温': 'temp_ambient',
    '方位角': 'azimuth',
    '云层不透明度': 'cloud_opacity',
    '露点温度': 'temp_dew_point',
    'DHI（太阳散射辐射指数）': 'dhi',
    'DNI（太阳直接辐射指数）': 'dni',
    'GHI（太阳总水平辐射）': 'ghi',
    'GTI（固定倾角辐射）': 'gti_fixed',
    'GTI（跟踪倾角辐射）': 'gti_tracking',
    '实际功率': 'active_power',
    '大气可降水量': 'precipitable_water',
    '相对湿度': 'humidity',
    '降雪深度': 'snow_depth',
    '地面气压': 'surface_pressure',
    '高度10m风向': 'wind_direction_10m',
    '高度10m风速': 'wind_speed_10m',
    '天顶角': 'zenith_angle'
}
df = df.rename(columns=RENAME_MAPPING)

df['timestamp_end'] = pd.to_datetime(df['timestamp_end'], errors='coerce')
df = df.sort_values('timestamp_end').reset_index(drop=True)

BASE_FEATURES = ['temp_ambient', 'ghi', 'gti_fixed', 'wind_speed_10m', 'humidity', 'active_power']
for col in BASE_FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce').interpolate(method='linear').fillna(0)

## Hourly aggregation

In [ ]:
NUMERIC_COLS = [
    'temp_ambient', 'azimuth', 'cloud_opacity', 'temp_dew_point',
    'dhi', 'dni', 'ghi', 'gti_fixed', 'gti_tracking', 'active_power',
    'precipitable_water', 'humidity', 'snow_depth', 'surface_pressure',
    'wind_direction_10m', 'wind_speed_10m', 'zenith_angle'
]
for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df_temp = df[['timestamp_end'] + NUMERIC_COLS].copy()
df_temp.set_index('timestamp_end', inplace=True)

df_hourly = df_temp.resample('h').mean().reset_index()

df_hourly[NUMERIC_COLS] = df_hourly[NUMERIC_COLS].interpolate(method='linear').bfill().ffill()

hours = df_hourly['timestamp_end'].dt.hour
months = df_hourly['timestamp_end'].dt.month
df_hourly['HOUR_SIN'] = np.sin(2 * np.pi * hours / 24.0)
df_hourly['HOUR_COS'] = np.cos(2 * np.pi * hours / 24.0)
df_hourly['MONTH_SIN'] = np.sin(2 * np.pi * months / 12.0)
df_hourly['MONTH_COS'] = np.cos(2 * np.pi * months / 12.0)

## Normalisation

In [ ]:
max_val = df_hourly['active_power'].max()

if max_val < 500:
    rated_capacity_kW = 150.0
    power_actual_w = df_hourly['active_power'] * 1000.0
else:
    rated_capacity_kW = 150.0
    power_actual_w = df_hourly['active_power']

rated_capacity_w = rated_capacity_kW * 1000.0
df_hourly['Power_Actual_W'] = power_actual_w
df_hourly['Power_PU'] = np.clip(power_actual_w / rated_capacity_w, 0.0, 1.0)

## Unified dataset

In [ ]:
df_hourly['utctime'] = df_hourly['timestamp_end']
df_hourly['GHI'] = df_hourly['ghi']
df_hourly['Irradiance_POA'] = df_hourly['gti_fixed']
df_hourly['Temp_Ambient'] = df_hourly['temp_ambient']
df_hourly['Wind_Speed'] = df_hourly['wind_speed_10m']
df_hourly['Humidity'] = df_hourly['humidity']

df_hourly['Latitude'] = 38.53
df_hourly['Longitude'] = 106.27
df_hourly['Tilt'] = 37.0

FINAL_COLUMNS = [
    'utctime', 'Power_Actual_W', 'Power_PU',
    'Irradiance_POA', 'GHI', 'Temp_Ambient', 'Wind_Speed', 'Humidity',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'Latitude', 'Longitude', 'Tilt'
]
df_aligned = df_hourly[FINAL_COLUMNS].copy()

## Save

In [ ]:
SAVE_PATH = OUT_DIR / 'Ningxia.csv'
df_aligned.to_csv(SAVE_PATH, index=False, encoding='utf-8')